# Python with PostgreSQL 기초

이 노트북은 `psycopg`를 사용해 PostgreSQL에 연결하고 SQL을 직접 실행하는 실습입니다. SQLAlchemy는 사용하지 않습니다.

## 1. 패키지 불러오기와 접속 정보 준비

상위 `docker-compose.yml`의 기본 접속 정보는 다음과 같습니다.

- host: `localhost`
- port: `5432`
- database: `examples_db`
- user: `admin`
- password: `admin123`

In [1]:
import os

import pandas as pd
import psycopg
from dotenv import load_dotenv
from psycopg.rows import dict_row

load_dotenv()

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
    "database": os.getenv("DB_NAME", "examples_db"),
    "user": os.getenv("DB_USER", "admin"),
    "password": os.getenv("DB_PASSWORD", "admin123"),
}

DB_CONFIG

{'host': 'localhost',
 'port': '5432',
 'database': 'examples_db',
 'user': 'admin',
 'password': 'admin123'}

## 2. PostgreSQL 연결 확인

`SELECT version()`을 실행해서 PostgreSQL 서버에 연결되는지 확인합니다.

In [2]:
conninfo = (
    f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

with psycopg.connect(conninfo, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version() AS version;")
        result = cur.fetchone()

result

{'version': 'PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit'}

## 3. 실습 테이블 만들기

Python 실습 전용 테이블을 만듭니다. 같은 노트북을 여러 번 실행해도 되도록 `IF NOT EXISTS`를 사용합니다.

In [3]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS python_students (
    student_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    email VARCHAR(120) NOT NULL UNIQUE,
    score NUMERIC(5, 2),
    created_at TIMESTAMPTZ NOT NULL DEFAULT now()
);

COMMENT ON TABLE python_students IS 'Python 연동 실습용 수강생 정보';
COMMENT ON COLUMN python_students.student_id IS '수강생을 식별하는 자동 증가 기본키';
COMMENT ON COLUMN python_students.name IS '수강생 이름';
COMMENT ON COLUMN python_students.email IS '수강생 이메일, 중복 불가';
COMMENT ON COLUMN python_students.score IS 'Python 실습용 점수';
COMMENT ON COLUMN python_students.created_at IS '수강생 등록 시각';
"""

with psycopg.connect(conninfo) as conn:
    with conn.cursor() as cur:
        cur.execute(create_table_sql)

print("python_students 테이블 준비 완료")

python_students 테이블 준비 완료


## 4. INSERT 실행하기

값을 SQL 문자열에 직접 붙이지 않고 `%s` 자리표시자와 파라미터를 사용합니다.

In [4]:
students = [
    ("김민준", "py_minjun@example.com", 92.5),
    ("이서연", "py_seoyeon@example.com", 85.0),
    ("박도윤", "py_doyun@example.com", 78.0),
]

insert_sql = """
INSERT INTO python_students (name, email, score)
VALUES (%s, %s, %s)
ON CONFLICT (email)
DO UPDATE SET
    name = EXCLUDED.name,
    score = EXCLUDED.score;
"""

with psycopg.connect(conninfo) as conn:
    with conn.cursor() as cur:
        cur.executemany(insert_sql, students)

print("데이터 입력 완료")

데이터 입력 완료


## 5. SELECT 결과 가져오기

`fetchall()`로 조회 결과를 가져온 뒤 pandas DataFrame으로 확인합니다.

In [5]:
select_sql = """
SELECT
    student_id,
    name,
    email,
    score,
    created_at
FROM python_students
ORDER BY student_id;
"""

with psycopg.connect(conninfo, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute(select_sql)
        rows = cur.fetchall()

pd.DataFrame(rows)

,student_id,name,email,score,created_at
0,1,김민준,py_minjun@example.com,92.50,2026-07-02 02:07:23.783680+00:00
1,2,이서연,py_seoyeon@example.com,85.00,2026-07-02 02:07:23.783680+00:00
2,3,박도윤,py_doyun@example.com,78.00,2026-07-02 02:07:23.783680+00:00


## 6. 조건을 사용해 조회하기

Python 변수 값을 SQL 조건에 전달합니다.

In [6]:
minimum_score = 80

with psycopg.connect(conninfo, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT name, email, score
            FROM python_students
            WHERE score >= %s
            ORDER BY score DESC;
            """,
            (minimum_score,),
        )
        rows = cur.fetchall()

pd.DataFrame(rows)

,name,email,score
0,김민준,py_minjun@example.com,92.50
1,이서연,py_seoyeon@example.com,85.00


## 7. UPDATE와 DELETE 실행하기

조건에 맞는 데이터를 수정하고 삭제합니다.

In [7]:
with psycopg.connect(conninfo, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            UPDATE python_students
            SET score = %s
            WHERE email = %s
            RETURNING student_id, name, email, score;
            """,
            (95.0, "py_minjun@example.com"),
        )
        updated_row = cur.fetchone()

updated_row

{'student_id': 1,
 'name': '김민준',
 'email': 'py_minjun@example.com',
 'score': Decimal('95.00')}

In [8]:
with psycopg.connect(conninfo, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            DELETE FROM python_students
            WHERE email = %s
            RETURNING student_id, name, email;
            """,
            ("py_doyun@example.com",),
        )
        deleted_row = cur.fetchone()

deleted_row

{'student_id': 3, 'name': '박도윤', 'email': 'py_doyun@example.com'}

## 8. postgresql.py 연결 클래스 사용해보기

강의 폴더에 있는 `postgresql.py`는 연결 객체를 재사용하기 위한 간단한 예제입니다.

In [9]:
from postgresql import PostgreDB

db = PostgreDB(DB_CONFIG)

with db.get_conn().cursor(row_factory=dict_row) as cur:
    cur.execute("SELECT COUNT(*) AS student_count FROM python_students;")
    count_row = cur.fetchone()

count_row

{'student_count': 2}

## 정리

- `psycopg.connect()`로 PostgreSQL에 연결합니다.
- `cursor.execute()`로 SQL을 실행합니다.
- 값은 SQL 문자열에 직접 붙이지 않고 파라미터로 전달합니다.
- 조회 결과는 `fetchone()`, `fetchall()`로 가져올 수 있습니다.
- pandas DataFrame을 사용하면 조회 결과를 표 형태로 보기 쉽습니다.